In [ ]:
# 허깅페이스 로그인
# 주피터 노트북 웹 UI 상에 로그인 위젯 띄우기
from huggingface_hub import notebook_login
notebook_login()

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer,pipeline, BitsAndBytesConfig
from langchain_huggingface import HuggingFacePipeline
hf_model = 'LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct'
# 4bit 양자화 설정(VRAM절약)
q_config =  BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)
# 토크나이져 , 모델 로드
model = AutoModelForCausalLM.from_pretrained(
    hf_model,
    quantization_config = q_config,
    device_map="auto",
    trust_remote_code=True
)
tokenizer = AutoTokenizer.from_pretrained(hf_model,trust_remote_code=True)

# 파이프라인생성 langchain llm 매핑
pipe =  pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.1,
    do_sample=True
)
exaone_llm = HuggingFacePipeline(pipeline=pipe)

[transformers] The `check_model_inputs` decorator is deprecated in favor of `merge_with_config_defaults`.


[ERROR] `cache_position` is part of ExaoneModel.forward's signature, but not documented. Make sure to add it to the docstring of the function in C:\Users\Playdata\.cache\huggingface\modules\transformers_modules\LGAI_hyphen_EXAONE\EXAONE_hyphen_3_dot_5_hyphen_2_dot_4B_hyphen_Instruct\ccce25bd39c141fe053e0bc75818a8f5fe962802\modeling_exaone.py.
[ERROR] `cache_position` is part of ExaoneForCausalLM.forward's signature, but not documented. Make sure to add it to the docstring of the function in C:\Users\Playdata\.cache\huggingface\modules\transformers_modules\LGAI_hyphen_EXAONE\EXAONE_hyphen_3_dot_5_hyphen_2_dot_4B_hyphen_Instruct\ccce25bd39c141fe053e0bc75818a8f5fe962802\modeling_exaone.py.


Exception in thread Thread-4 (_readerthread):
Traceback (most recent call last):
  File "c:\Users\Playdata\miniconda3\envs\torch_env\lib\threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "c:\Users\Playdata\miniconda3\envs\torch_env\lib\threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\Playdata\miniconda3\envs\torch_env\lib\subprocess.py", line 1515, in _readerthread
    buffer.append(fh.read())
  File "c:\Users\Playdata\miniconda3\envs\torch_env\lib\codecs.py", line 322, in decode
    (result, consumed) = self._buffer_decode(data, self.errors, final)
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc0 in position 6: invalid start byte


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

c:\Users\Playdata\miniconda3\envs\torch_env\lib\site-packages\bitsandbytes\backends\default\ops.py:223: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
c:\Users\Playdata\miniconda3\envs\torch_env\lib\site-packages\bitsandbytes\backends\cpu\ops.py:36: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


...

vocab.json:   0%|          | 0.00/1.93M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/563 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.96M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [ ]:
# 로컬모델은 특수기호에 민감해.. 지키지 않으면 환각
from langchain_core.prompts import PromptTemplate
exaone_template = '''[|system|]
당신은 사내 문서를 바탕으로 질문에 답하는 어시스턴트입니다.
아래 [문서] 내용만 참고하고, 없으면 '모릅니다'라고 하세요 [|endofturn|]
[|user|]
[문서]
{context}

질문: {question}[|endofturn|]
[|assistant|]
'''
exaone_prompt = PromptTemplate(
    input_variables=['context','question'],
    template=exaone_template
)
chain = exaone_prompt | exaone_llm | Strout